# 1.讀取atp_players.csv

In [1]:
import pandas as pd

players = pd.read_csv("./dataset/HW2/JEFFSM_tennis_atp-master/atp_players.csv")
print(f"欄位：{players.columns.tolist()}")

欄位：['player_id', 'name_first', 'name_last', 'hand', 'dob', 'ioc', 'height', 'wikidata_id']


C:\Users\boss9\AppData\Local\Temp\ipykernel_23344\3447816942.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  players = pd.read_csv("./dataset/HW2/JEFFSM_tennis_atp-master/atp_players.csv")


In [2]:
print(f"球員數量：{len(players)}")

球員數量：65989


In [3]:
print(f"生日資料不完整數量：{players['dob'].isna().sum()}")


生日資料不完整數量：18402


In [4]:
counts = players["ioc"].value_counts()
print(f"最多球員的國家：{counts.index[0]}，共{counts.iloc[0]}人")

最多球員的國家：USA，共13228人


In [5]:
tpe = players["ioc"].value_counts().reindex(["TPE"])
twn = players["ioc"].value_counts().reindex(["TWN"])
print(f"TPE球員數量：{tpe.iloc[0]}，TWN球員數量：{twn.iloc[0]}")

TPE球員數量：201，TWN球員數量：1


In [6]:
name = players.loc[players["ioc"] == "TPE", "name_last"].dropna().unique()
print(f"TPE球員姓氏：{'、'.join(name)}")

TPE球員姓氏：Lien、Chen、Lin、Tsai、Su、Cheng、Lu、Wang、Lee、Chang、Yi、Jung、Peng、Hung、Yang、Hsu、Hsieh、Mao、Huang、Liao、Ho、Yu、Jao、Liu、Shen、Wu、Menn、Tang、Pan、Lai、Shieh、Chuang、Liang、Yeh、Kuo、Hsueh、Chien、Hu、Chiu、Ko、Lo、Wen、Tan、Meng、Ballout、Kao、Wei、Li、Sung、Shih、Chan、Chiang、Chu、Yin、Tseng、He、Shie、Chung、Pai、Han、Fu、Wong、Ou、Hou


# 2.讀取atp_matches*202[1234].csv

In [7]:
from pathlib import Path

folder = Path("./dataset/HW2/JEFFSM_tennis_atp-master")
files = sorted(folder.glob("atp_matches*202[1234].csv"))
for file in files:
    print(file.name)

atp_matches_2021.csv
atp_matches_2022.csv
atp_matches_2023.csv
atp_matches_2024.csv
atp_matches_futures_2021.csv
atp_matches_futures_2022.csv
atp_matches_futures_2023.csv
atp_matches_futures_2024.csv
atp_matches_qual_chall_2021.csv
atp_matches_qual_chall_2022.csv
atp_matches_qual_chall_2023.csv
atp_matches_qual_chall_2024.csv


In [8]:
matches = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
wins = (matches["winner_ioc"] == "TPE").sum()
losses = (matches["loser_ioc"] == "TPE").sum()
print(f"TPE球員贏了{wins}場，輸了{losses}場")

TPE球員贏了732場，輸了765場


In [9]:
levels = matches["tourney_level"].dropna().astype(str).unique()
print(f"共有{len(levels)}種比賽層級：{'、'.join(sorted(levels))}")

共有9種比賽層級：15、25、A、C、D、F、G、M、O


In [10]:
height_matches = matches.dropna(subset=["winner_ht", "loser_ht"]).copy()
height_matches["tourney_level"] = (height_matches["tourney_level"].astype(str))
height_matches["身高關係"] = "等高"
height_matches.loc[height_matches["winner_ht"] > height_matches["loser_ht"], "身高關係"] = "勝者較高"
height_matches.loc[height_matches["winner_ht"] < height_matches["loser_ht"], "身高關係"] = "勝者較矮"
height_table = pd.crosstab(height_matches["tourney_level"],
                           height_matches["身高關係"]).reindex(columns=["勝者較高","等高", "勝者較矮"], fill_value=0)
print("winner loser身高比較：")
height_table

winner loser身高比較：


身高關係,勝者較高,等高,勝者較矮
tourney_level,,,
15,9657,2205,9092
25,7461,1824,7019
A,4131,1009,3629
C,14372,3629,13772
D,303,90,284
F,33,10,32
G,1859,427,1523
M,1636,389,1310
O,29,4,31


In [11]:
age_matches = matches.copy()
age_matches["winner_age"] = (age_matches["winner_age"].round().astype("Int64"))
age_matches["loser_age"] = (age_matches["loser_age"].round().astype("Int64"))
winner_age_counts = (age_matches["winner_age"].value_counts().sort_index())
print("winner年紀勝場統計：")
winner_age_counts

winner年紀勝場統計：


winner_age
14        1
15       17
16      167
17      748
18     3109
19     5565
20     9032
21     8449
22    11317
23    10245
24    12604
25    10069
26    10241
27     6981
28     6404
29     4286
30     4297
31     3000
32     2627
33     1598
34     1564
35      976
36      791
37      400
38      275
39       60
40       54
41       10
42       12
43        3
44        4
45        2
Name: count, dtype: Int64

In [12]:
wins_by_age = age_matches["winner_age"].value_counts()
losses_by_age = age_matches["loser_age"].value_counts()
age_stats = pd.DataFrame({"勝場": wins_by_age, "敗場": losses_by_age}).fillna(0).astype(int)
age_stats.index.name = "年齡"
age_stats = age_stats.sort_index()
age_stats["出賽場數"] = age_stats["勝場"] + age_stats["敗場"]
age_stats["勝率"] = (age_stats["勝場"] / age_stats["出賽場數"])
print("球員年齡統計勝率：")
age_stats.style.format({"勝率": "{:.2%}"})

球員年齡統計勝率：


,勝場,敗場,出賽場數,勝率
年齡,,,,
14,1,6,7,14.29%
15,17,58,75,22.67%
16,167,392,559,29.87%
17,748,1269,2017,37.08%
18,3109,4487,7596,40.93%
19,5565,6600,12165,45.75%
20,9032,9145,18177,49.69%
21,8449,8034,16483,51.26%
22,11317,10916,22233,50.90%


In [13]:
top10_wins = (matches.groupby("winner_id").agg(姓名=("winner_name", "first"), 勝場=("winner_id", "size"))
              .sort_values("勝場", ascending=False).head(10))
print("top 10勝場數球員")
top10_wins

top 10勝場數球員


,姓名,勝場
winner_id,,
206173,Jannik Sinner,237
207660,Nicholas David Ionel,232
207989,Carlos Alcaraz,224
106421,Daniil Medvedev,221
144826,Aziz Dougaz,213
208029,Holger Rune,213
100644,Alexander Zverev,213
126774,Stefanos Tsitsipas,212
126094,Andrey Rublev,205


In [14]:
player_wins = matches.groupby("winner_id").size()
player_losses = matches.groupby("loser_id").size()
player_stats = pd.DataFrame({"勝場": player_wins, "敗場": player_losses}).fillna(0).astype(int)
player_stats.index.name = "球員ID"
names = pd.concat([matches[["winner_id", "winner_name"]].rename(columns={"winner_id": "球員ID", "winner_name": "姓名"}),
                   matches[["loser_id", "loser_name"]].rename(columns={"loser_id": "球員ID", "loser_name": "姓名"})
                   ]).drop_duplicates("球員ID").set_index("球員ID")
player_stats["姓名"] = names["姓名"]
player_stats["出賽場數"] = player_stats["勝場"] + player_stats["敗場"]
player_stats["勝率"] = player_stats["勝場"] / player_stats["出賽場數"]
top10_rate = player_stats.sort_values(["勝率", "勝場"],ascending=[False, False]).head(10)
print("top 10勝場率球員")
top10_rate[["姓名", "勝場", "敗場", "出賽場數", "勝率"]].style.format({"勝率": "{:.2%}"})

top 10勝場率球員


,姓名,勝場,敗場,出賽場數,勝率
球員ID,,,,,
104235,Matwe Middelkoop,1,0,1,100.00%
208414,Sylvestre Monnou,1,0,1,100.00%
212721,Benedict Badza,1,0,1,100.00%
104925,Novak Djokovic,193,31,224,86.16%
105656,Marcelo Arevalo,8,2,10,80.00%
207989,Carlos Alcaraz,224,57,281,79.72%
207985,Jacob Fearnley,56,15,71,78.87%
206173,Jannik Sinner,237,64,301,78.74%
202475,Philip Henning,74,23,97,76.29%


In [15]:
age_19_28 = age_matches.loc[age_matches["winner_age"].between(19, 28) & age_matches["loser_age"].between(19, 28)]
age_count_table = pd.crosstab(age_19_28["winner_age"], age_19_28["loser_age"])
age_count_table = age_count_table.reindex(index=range(19, 29), columns=range(19, 29), fill_value=0)
print("不同年紀球員之間的對戰數量(19~28歲)")
age_count_table

不同年紀球員之間的對戰數量(19~28歲)


loser_age,19,20,21,22,23,24,25,26,27,28
winner_age,,,,,,,,,,
19,416,524,414,561,493,576,456,448,279,249
20,608,845,703,898,812,939,663,756,432,408
21,564,706,637,884,733,837,657,695,456,418
22,726,944,825,1211,948,1091,919,901,607,565
23,626,854,738,929,848,1086,859,842,553,512
24,740,1013,909,1148,1054,1390,1047,1086,649,675
25,554,800,718,928,833,1018,775,850,614,555
26,564,776,641,925,896,1098,839,829,565,579
27,342,553,447,656,547,697,612,616,405,400


In [16]:
total_games = age_count_table + age_count_table.T
age_win_rate = age_count_table / total_games.replace(0, float("nan"))
age_win_rate.index.name = "球員年齡"
age_win_rate.columns.name = "對手年齡"
print("不同年紀球員之間的對戰數量(勝率)")
age_win_rate.style.format("{:.2%}", na_rep="無交手")

不同年紀球員之間的對戰數量(勝率)


對手年齡,19,20,21,22,23,24,25,26,27,28
球員年齡,,,,,,,,,,
19,50.00%,46.29%,42.33%,43.59%,44.06%,43.77%,45.15%,44.27%,44.93%,40.55%
20,53.71%,50.00%,49.89%,48.75%,48.74%,48.10%,45.32%,49.35%,43.86%,45.13%
21,57.67%,50.11%,50.00%,51.73%,49.83%,47.94%,47.78%,52.02%,50.50%,51.29%
22,56.41%,51.25%,48.27%,50.00%,50.51%,48.73%,49.76%,49.34%,48.06%,50.13%
23,55.94%,51.26%,50.17%,49.49%,50.00%,50.75%,50.77%,48.45%,50.27%,50.84%
24,56.23%,51.90%,52.06%,51.27%,49.25%,50.00%,50.70%,49.73%,48.22%,50.87%
25,54.85%,54.68%,52.22%,50.24%,49.23%,49.30%,50.00%,50.33%,50.08%,51.58%
26,55.73%,50.65%,47.98%,50.66%,51.55%,50.27%,49.67%,50.00%,47.84%,51.28%
27,55.07%,56.14%,49.50%,51.94%,49.73%,51.78%,49.92%,52.16%,50.00%,52.98%


In [17]:
qualified_ages = []
for age in age_win_rate.index:
    rates = age_win_rate.loc[age].drop(index=age)
    if rates.notna().all() and (rates >= 0.49).all():
        qualified_ages.append(age)
print(
    "19~28歲選手中，對手是其他年齡層的勝率在49%以上，符合條件的年齡層："
    + ("、".join(map(str, qualified_ages)) if qualified_ages else "無")
)

19~28歲選手中，對手是其他年齡層的勝率在49%以上，符合條件的年齡層：25、27


# 3.五個自行設計的問題

In [18]:
top10_min10 = (player_stats.loc[player_stats["出賽場數"] >= 10].sort_values(["勝率", "勝場"], ascending=[False, False]).head(10))
print("top 10勝場率球員(出場次數小於10場不採計)")
top10_min10[["姓名", "勝場", "敗場", "出賽場數", "勝率"]].style.format({"勝率": "{:.2%}"})

top 10勝場率球員(出場次數小於10場不採計)


,姓名,勝場,敗場,出賽場數,勝率
球員ID,,,,,
104925,Novak Djokovic,193,31,224,86.16%
105656,Marcelo Arevalo,8,2,10,80.00%
207989,Carlos Alcaraz,224,57,281,79.72%
207985,Jacob Fearnley,56,15,71,78.87%
206173,Jannik Sinner,237,64,301,78.74%
202475,Philip Henning,74,23,97,76.29%
106421,Daniil Medvedev,221,71,292,75.68%
104745,Rafael Nadal,77,25,102,75.49%
210530,Learner Tien,106,35,141,75.18%


In [19]:
left_wins = ((matches["winner_hand"] == "L") & (matches["loser_hand"] == "R")).sum()
left_losses = ((matches["winner_hand"] == "R") & (matches["loser_hand"] == "L")).sum()
total = left_wins + left_losses
print(f"左右手對戰共{total}場")
print(f"左手球員勝場：{left_wins}，敗場：{left_losses}")
if total > 0:
    print(f"左手球員勝率：{left_wins / total:.2%}")
else:
    print("沒有符合條件的比賽")

左右手對戰共22368場
左手球員勝場：11197，敗場：11171
左手球員勝率：50.06%


In [20]:
rank_matches = matches.copy()
for column in ["winner_rank", "loser_rank"]:
    rank_matches[column] = pd.to_numeric(rank_matches[column], errors="coerce")
rank_matches = rank_matches.dropna(subset=["surface", "winner_rank", "loser_rank"]).copy()
rank_matches = rank_matches.loc[(rank_matches["winner_rank"] > 0) & (rank_matches["loser_rank"] > 0)].copy()
rank_matches["爆冷"] = (rank_matches["winner_rank"] > rank_matches["loser_rank"])
upset_table = rank_matches.groupby("surface").agg(比賽場數=("爆冷", "size"), 爆冷場數=("爆冷", "sum"))
upset_table["爆冷比例"] = (upset_table["爆冷場數"] / upset_table["比賽場數"])
print("不同場地中，排名較低的球員爆冷獲勝的比例")
upset_table.sort_values("爆冷比例", ascending=False).style.format({"爆冷比例": "{:.2%}"})

不同場地中，排名較低的球員爆冷獲勝的比例


,比賽場數,爆冷場數,爆冷比例
surface,,,
Grass,2812,1032,36.70%
Hard,51119,18514,36.22%
Carpet,588,209,35.54%
Clay,46250,16415,35.49%


In [ ]:
winner_records = matches[["winner_id", "winner_name", "surface"]].rename(columns={
    "winner_id": "球員ID", "winner_name": "姓名"}).assign(獲勝=1)
loser_records = matches[["loser_id", "loser_name", "surface"]].rename(columns={
    "loser_id": "球員ID", "loser_name": "姓名"}).assign(獲勝=0)
player_records = pd.concat([winner_records, loser_records], ignore_index=True).dropna(subset=["球員ID", "surface"])
surface_stats = player_records.groupby(["球員ID", "surface"]).agg(
    姓名=("姓名", "first"), 出賽場數=("獲勝", "size"), 勝場=("獲勝", "sum"))
surface_stats["勝率"] = (surface_stats["勝場"] / surface_stats["出賽場數"])
eligible_surfaces = surface_stats.loc[surface_stats["出賽場數"] >= 10].reset_index()
surface_gap = eligible_surfaces.groupby("球員ID").agg(姓名=("姓名", "first"), 符合場地數=("surface", "nunique"),
                                                    最高勝率=("勝率", "max"), 最低勝率=("勝率", "min"))
surface_gap = surface_gap.loc[surface_gap["符合場地數"] >= 2].copy()
surface_gap["勝率差距"] = (surface_gap["最高勝率"] - surface_gap["最低勝率"])
print("至少在兩種場地各出賽10場的球員中，誰的場地勝率差距最大")
surface_gap.sort_values("勝率差距", ascending=False).head(10).style.format({"最高勝率": "{:.2%}", "最低勝率": "{:.2%}", "勝率差距": "{:.2%}"})

至少在兩種場地各出賽10場的球員中，誰的場地勝率差距最大


,姓名,符合場地數,最高勝率,最低勝率,勝率差距
球員ID,,,,,
111153,Christopher Eubanks,3,62.86%,14.29%,48.57%
123828,Jan Choinski,3,64.29%,18.18%,46.10%
208363,Mariano Navone,2,64.19%,18.75%,45.44%
202104,Sebastian Baez,3,72.16%,27.27%,44.89%
106065,Marco Cecchinato,2,55.67%,11.11%,44.55%
105173,Adrian Mannarino,3,53.85%,10.34%,43.50%
210447,Leo Raquillet,2,43.33%,0.00%,43.33%
200309,Marek Gengel,2,69.73%,27.03%,42.70%
200385,Dan Added,3,84.00%,42.19%,41.81%


In [24]:
year_matches = matches.copy()
year_matches["年份"] = pd.to_datetime(year_matches["tourney_date"].astype("string"), format="%Y%m%d", errors="coerce").dt.year
tpe_wins = (year_matches.loc[year_matches["winner_ioc"] == "TPE"].groupby("年份").size())
tpe_losses = ( year_matches.loc[year_matches["loser_ioc"] == "TPE"].groupby("年份").size())
tpe_year_stats = (pd.DataFrame({"勝場": tpe_wins, "敗場": tpe_losses}).reindex(range(2021, 2025)).fillna(0).astype(int))
tpe_year_stats.index.name = "年份"
tpe_year_stats["出賽場數"] = (tpe_year_stats["勝場"] + tpe_year_stats["敗場"])
tpe_year_stats["勝率"] = (tpe_year_stats["勝場"] / tpe_year_stats["出賽場數"].replace(0, float("nan")))
print("TPE球員在2021～2024年，每年的出賽場數和勝率如何變化")
tpe_year_stats.style.format({"勝率": "{:.2%}"}, na_rep="無出賽")

TPE球員在2021～2024年，每年的出賽場數和勝率如何變化


,勝場,敗場,出賽場數,勝率
年份,,,,
2021,139,150,289,48.10%
2022,210,200,410,51.22%
2023,202,206,408,49.51%
2024,181,209,390,46.41%
